# 4장 웹부록 | 토큰을 세고 임베딩 지도 그리기

「영화로 보는 AI 리터러시」 4장 '말하는 AI: 대규모 언어 모델'의 코랩 실습입니다.

세 가지를 해 봅니다.
1. 같은 문장을 한국어와 영어로 넣어 **토큰**이 몇 개로 잘리는지 세어 봅니다.
2. 낱말 수십 개의 **임베딩**을 평면 지도로 그리고, '왕 − 남자 + 여자'를 직접 계산합니다.
3. 실습 2의 **주사위 언어 모델**을 컴퓨터로 만들어, 앞 낱말을 하나 볼 때와 셋 볼 때를 비교합니다.

진짜 언어 모델은 너무 커서 코랩에서 내려받기 어렵습니다. 그래서 여기서는 모든 것을 **작은 장난감 크기로 직접** 만듭니다. 크기만 작을 뿐 원리는 책에서 배운 것과 같습니다.

**사용법**
- 위에서부터 차례로 각 칸 왼쪽의 ▶(실행) 버튼을 누르세요. 코드를 몰라도 괜찮습니다.
- `# ← 바꿔 보세요`라고 적힌 곳만 고쳐서 다시 ▶를 누르면 됩니다.
- 처음부터 다시 하고 싶으면 메뉴에서 **런타임 → 모두 실행**을 누르세요.

## 0단계. 그래프에 한글 글꼴 준비하기

코랩에는 한글 글꼴이 없어서 그래프의 한글이 네모(□)로 깨집니다. 이 칸이 나눔고딕 글꼴을 설치합니다(10초쯤 걸립니다).
설치하지 못하면 그래프의 낱말을 영어로 대신 적습니다. 어느 쪽이든 실습은 그대로 됩니다.

In [ ]:
import sys, subprocess
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

def find_korean_font():
    for f in fm.findSystemFonts():
        if any(k in f for k in ("NanumGothic.ttf", "NotoSansCJK", "NotoSansKR", "AppleGothic", "malgun")):
            return f
    return None

KFONT = find_korean_font()
if KFONT is None and "google.colab" in sys.modules:
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
    KFONT = find_korean_font()
if KFONT:
    fm.fontManager.addfont(KFONT)
    plt.rcParams["font.family"] = fm.FontProperties(fname=KFONT).get_name()
    plt.rcParams["axes.unicode_minus"] = False
    print("한글 글꼴 준비 완료:", KFONT)
else:
    print("한글 글꼴을 찾지 못해 그래프 속 낱말은 영어로 표시합니다.")

## 1단계. 글자는 컴퓨터 안에서 바이트가 된다

1장에서 본 것처럼 컴퓨터는 모든 것을 숫자로 바꿉니다. 글자를 숫자로 적는 약속(UTF-8)에서 영어 글자는 1바이트, 한글 글자는 3바이트입니다.
요즘 언어 모델의 토큰은 이 바이트에서 출발합니다.

In [ ]:
for word in ["movie", "영화", "AI", "인공지능"]:
    b = word.encode("utf-8")
    print(f"{word:6s} → 글자 {len(word)}개, 바이트 {len(b):2d}개: {list(b)}")

## 2단계. 작은 토크나이저 학습시키기

토크나이저는 글을 토큰으로 자르는 도구입니다. 많이 쓰이는 방법(BPE)은 이렇습니다.

1. 처음에는 모든 글을 바이트 하나하나로 자릅니다.
2. 학습 글에서 **가장 자주 붙어 나오는 두 조각**을 찾아 한 덩어리로 합칩니다.
3. 이것을 수백 번 되풀이합니다. 그러면 자주 쓰는 말은 한 덩어리가 되고, 드문 말은 잘게 남습니다(3-2절).

초기의 언어 모델처럼 **영어 글을 훨씬 많이** 넣어 학습시켜 봅니다. 아래 학습 글은 같은 내용을 영어와 한국어로 적은 것인데, 영어를 9번, 한국어를 1번 넣습니다.

In [ ]:
from collections import Counter

PARALLEL = [
    ("The movie tells the story of a lonely man.", "영화는 외로운 남자의 이야기를 들려줍니다."),
    ("He falls in love with an artificial intelligence.", "그는 인공지능과 사랑에 빠집니다."),
    ("She talks to him every day.", "그녀는 매일 그에게 말을 겁니다."),
    ("The weather is very nice today.", "오늘 날씨가 정말 좋습니다."),
    ("I want to watch a movie with my friends.", "나는 친구들과 영화를 보고 싶습니다."),
    ("Language models learn which word comes next.", "언어 모델은 다음에 올 낱말을 배웁니다."),
    ("A token is a small piece of text.", "토큰은 글의 작은 조각입니다."),
    ("The student asked the computer a question.", "학생이 컴퓨터에게 질문을 했습니다."),
    ("The computer gave a long answer.", "컴퓨터는 긴 답을 주었습니다."),
    ("Artificial intelligence is changing how we study.", "인공지능은 우리가 공부하는 방식을 바꾸고 있습니다."),
    ("Many students watch movies about robots.", "많은 학생이 로봇에 관한 영화를 봅니다."),
    ("My friend and I talked about the movie after school.", "친구와 나는 방과 후에 영화 이야기를 했습니다."),
    ("The model reads the sentence and guesses the next word.", "모델은 문장을 읽고 다음 낱말을 맞힙니다."),
    ("People use chat programs to write letters.", "사람들은 편지를 쓰려고 채팅 프로그램을 씁니다."),
    ("Can machines think like people?", "기계는 사람처럼 생각할 수 있을까요?"),
]
EN_TEXT = "\n".join(e for e, k in PARALLEL)
KO_TEXT = "\n".join(k for e, k in PARALLEL)


def train_bpe(text, n_merges):
    """가장 자주 붙어 나오는 두 조각을 합치는 일을 n_merges번 되풀이합니다."""
    words = Counter(tuple((" " + w).encode("utf-8")) for w in text.split())
    rules = []
    for _ in range(n_merges):
        pairs = Counter()
        for w, c in words.items():
            for a, b in zip(w, w[1:]):
                pairs[(a, b)] += c
        if not pairs:
            break
        (a, b), c = pairs.most_common(1)[0]
        if c < 2:
            break
        rules.append((a, b))
        merged = Counter()
        for w, cnt in words.items():
            out, i = [], 0
            while i < len(w):
                if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                    out.append((a, b)); i += 2
                else:
                    out.append(w[i]); i += 1
            merged[tuple(out)] += cnt
        words = merged
    return rules


def to_bytes(t):
    return bytes([t]) if isinstance(t, int) else to_bytes(t[0]) + to_bytes(t[1])


def tokenize(text, rules):
    rank = {r: i for i, r in enumerate(rules)}
    tokens = []
    for w in text.split():
        w = list((" " + w).encode("utf-8"))
        while True:
            best = None
            for i, pair in enumerate(zip(w, w[1:])):
                r = rank.get(pair)
                if r is not None and (best is None or r < best[0]):
                    best = (r, i)
            if best is None:
                break
            i = best[1]
            w = w[:i] + [(w[i], w[i + 1])] + w[i + 2:]
        tokens += [to_bytes(t) for t in w]
    return tokens


def show(tokens):
    """토큰을 | 로 나눠 보여 줍니다. 글자가 되지 못한 바이트 조각은 ? 로 표시합니다."""
    pieces = []
    for t in tokens:
        try:
            pieces.append(t.decode("utf-8").replace(" ", "_"))
        except UnicodeDecodeError:
            pieces.append("?")
    return "|".join(pieces)


N_MERGES = 300   # ← 합치기 횟수(어휘 크기)를 바꿔 보세요
rules_en = train_bpe((EN_TEXT + "\n") * 9 + KO_TEXT, N_MERGES)   # 영어 9 : 한국어 1
print(f"합치기 규칙 {len(rules_en)}개를 배웠습니다. 처음 배운 규칙 10개:")
print([show([to_bytes(a) + to_bytes(b)]) for a, b in rules_en[:10]])

## 3단계. 같은 문장, 다른 토큰 수

이제 학습 글에 **없던** 새 문장을 한국어와 영어로 넣고 토큰 수를 셉니다. `?`는 한 글자도 되지 못한 바이트 조각입니다. `_`는 띄어쓰기 자리입니다.

👉 **바꿔 볼 곳**: `MY_KO`, `MY_EN`에 같은 뜻의 문장을 직접 넣어 보세요.

In [ ]:
MY_KO = "날씨가 좋아서 영화를 보고 싶습니다."               # ← 한국어 문장을 바꿔 보세요
MY_EN = "The weather is nice, so I want to watch a movie."    # ← 같은 뜻의 영어 문장

TESTS = [
    (MY_KO, MY_EN),
    ("학생은 매일 컴퓨터에게 말을 겁니다.", "The student talks to the computer every day."),
    ("인공지능은 질문을 읽고 답을 줍니다.", "Artificial intelligence reads the question and gives an answer."),
]

def compare(rules, title):
    print(f"[{title}]")
    total_ko = total_en = 0
    for ko, en in TESTS:
        tk, te = tokenize(ko, rules), tokenize(en, rules)
        total_ko += len(tk); total_en += len(te)
        print(f"  한국어 {len(tk):2d}토큰: {show(tk)}")
        print(f"  영어   {len(te):2d}토큰: {show(te)}\n")
    print(f"  합계: 한국어 {total_ko}토큰, 영어 {total_en}토큰 → 한국어가 {total_ko / total_en:.1f}배\n")
    return total_ko, total_en

ko1, en1 = compare(rules_en, "영어 위주로 학습한 토크나이저")

같은 뜻인데 한국어가 훨씬 많은 토큰으로 잘립니다. 학습 글에서 거의 못 본 한글은 글자 하나도 되지 못하고 바이트 조각(`?`)으로 잘게 쪼개집니다. 토큰이 많으면 요금도 더 내고, 한 번에 읽을 수 있는 글(컨텍스트 창)도 줄어듭니다.

이번에는 **한국어와 영어를 같은 양**으로 넣어 토크나이저를 다시 학습시킵니다. 요즘 모델들이 여러 언어를 고르게 다루도록 토큰을 다시 설계한 것과 같습니다(3-2절 '하나 더 알아가기').

In [ ]:
rules_bal = train_bpe((EN_TEXT + "\n") * 5 + (KO_TEXT + "\n") * 5, N_MERGES)   # 영어 5 : 한국어 5
ko2, en2 = compare(rules_bal, "한국어와 영어를 고르게 학습한 토크나이저")
print("※ 합치기 규칙의 수(어휘 크기)는 똑같이 정해 두었습니다. 한국어에 규칙을 나눠 주니 영어 토큰은 조금 늘었습니다.")

plt.figure(figsize=(6, 3.5))
x = [0, 1]
plt.bar([i - 0.2 for i in x], [ko1, ko2], width=0.4, label="Korean", color="#e8743b")
plt.bar([i + 0.2 for i in x], [en1, en2], width=0.4, label="English", color="#5b8def")
plt.xticks(x, ["English-heavy (9:1)", "Balanced (5:5)"])
plt.ylabel("Total tokens")
plt.title("Same sentences, different token counts")
plt.legend()
plt.show()

## 4단계. 임베딩: 낱말을 좌표로 바꾸기

토큰 번호만으로는 뜻을 알 수 없습니다. 그래서 낱말마다 숫자 여러 개로 된 **좌표**를 줍니다(3-3절).
좌표는 '어떤 낱말과 함께 자주 나오는가'에서 배웁니다. 비슷한 자리에 쓰이는 낱말은 뜻도 비슷하다는 생각입니다.

아래 칸은 작은 글 모음을 만듭니다. 사람, 도시와 나라, 과일, 동물, 탈것에 관한 짧은 문장 수천 개입니다. 예를 들어 '왕'은 '궁전', '왕관을', '그는' 같은 말과 함께 나옵니다.

In [ ]:
import random
import numpy as np

WORDS = {   # 낱말: (무리, 함께 자주 나오는 말의 종류들)
    "왕": ("사람", ["남", "왕족", "어른"]), "여왕": ("사람", ["여", "왕족", "어른"]),
    "왕자": ("사람", ["남", "왕족", "아이"]), "공주": ("사람", ["여", "왕족", "아이"]),
    "남자": ("사람", ["남", "평민", "어른"]), "여자": ("사람", ["여", "평민", "어른"]),
    "소년": ("사람", ["남", "평민", "아이"]), "소녀": ("사람", ["여", "평민", "아이"]),
    "서울": ("도시·나라", ["한국", "도시"]), "도쿄": ("도시·나라", ["일본", "도시"]),
    "파리": ("도시·나라", ["프랑스", "도시"]), "베이징": ("도시·나라", ["중국", "도시"]),
    "한국": ("도시·나라", ["한국", "국가"]), "일본": ("도시·나라", ["일본", "국가"]),
    "프랑스": ("도시·나라", ["프랑스", "국가"]), "중국": ("도시·나라", ["중국", "국가"]),
    "사과": ("과일", ["과일", "빨강"]), "딸기": ("과일", ["과일", "빨강"]), "바나나": ("과일", ["과일", "노랑"]),
    "귤": ("과일", ["과일", "노랑"]), "포도": ("과일", ["과일", "보라"]),
    "고양이": ("동물", ["동물", "반려"]), "강아지": ("동물", ["동물", "반려"]), "토끼": ("동물", ["동물", "반려"]),
    "호랑이": ("동물", ["동물", "맹수"]), "사자": ("동물", ["동물", "맹수"]), "곰": ("동물", ["동물", "맹수"]),
    "자동차": ("탈것", ["탈것", "도로"]), "버스": ("탈것", ["탈것", "도로"]), "자전거": ("탈것", ["탈것", "도로"]),
    "기차": ("탈것", ["탈것", "철도"]), "지하철": ("탈것", ["탈것", "철도"]), "비행기": ("탈것", ["탈것", "하늘"]),
}
CONTEXT = {
    "남": ["그는", "수염을", "아들", "형"], "여": ["그녀는", "치마를", "딸", "언니"],
    "왕족": ["궁전", "왕관을", "다스린다", "성에"], "평민": ["마을", "시장에", "일한다", "장터"],
    "어른": ["어른", "결혼한", "출근한다"], "아이": ["어린", "학교에", "논다"],
    "한국": ["김치", "한글", "원화"], "일본": ["스시", "엔화", "후지산"],
    "프랑스": ["바게트", "유로", "에펠탑"], "중국": ["만두", "위안화", "만리장성"],
    "도시": ["수도", "시청", "인구"], "국가": ["국민", "대통령", "국경"],
    "과일": ["달다", "먹는다", "과일", "껍질"], "빨강": ["빨갛다"], "노랑": ["노랗다"], "보라": ["보랏빛"],
    "동물": ["동물", "털", "꼬리"], "반려": ["집에서", "귀엽다"], "맹수": ["사납다", "숲속"],
    "탈것": ["탄다", "빠르다", "운전"], "도로": ["바퀴", "도로"], "철도": ["역", "선로"], "하늘": ["공항", "날개"],
}
EN_NAME = {"왕": "king", "여왕": "queen", "왕자": "prince", "공주": "princess", "남자": "man", "여자": "woman",
           "소년": "boy", "소녀": "girl", "서울": "Seoul", "도쿄": "Tokyo", "파리": "Paris", "베이징": "Beijing",
           "한국": "Korea", "일본": "Japan", "프랑스": "France", "중국": "China", "사과": "apple", "딸기": "strawberry",
           "바나나": "banana", "귤": "tangerine", "포도": "grape", "고양이": "cat", "강아지": "puppy", "토끼": "rabbit",
           "호랑이": "tiger", "사자": "lion", "곰": "bear", "자동차": "car", "버스": "bus", "자전거": "bicycle",
           "기차": "train", "지하철": "subway", "비행기": "airplane",
           "사람": "people", "도시·나라": "city/country", "과일": "fruit", "동물": "animal", "탈것": "vehicle"}
def label(w):
    return w if KFONT else EN_NAME.get(w, w)

OWN = {   # 낱말마다 저만의 짝꿍 말도 조금씩 섞습니다
    "사과": "아삭", "딸기": "케이크", "바나나": "원숭이", "귤": "겨울", "포도": "송이",
    "고양이": "야옹", "강아지": "멍멍", "토끼": "당근", "호랑이": "줄무늬", "사자": "갈기", "곰": "꿀",
    "자동차": "주차", "버스": "정류장", "자전거": "페달", "기차": "기적", "지하철": "환승", "비행기": "여권",
}

rng = random.Random(0)
sentences = []
for w, (group, kinds) in WORDS.items():
    for _ in range(60):
        ctx = [rng.choice(CONTEXT[rng.choice(kinds)]) for _ in range(2)]
        if w in OWN and rng.random() < 0.7:
            ctx[1] = OWN[w]
        sentences.append([w] + ctx)
print(f"문장 {len(sentences):,}개를 만들었습니다. 몇 개만 보면:")
for s in rng.sample(sentences, 6):
    print("  ", " ".join(s))

이제 '함께 나온 횟수' 표를 만들고, 그 표를 숫자 50개짜리 좌표로 줄입니다. 이것이 낱말의 임베딩입니다.

In [ ]:
vocab = sorted({t for s in sentences for t in s})
ix = {w: i for i, w in enumerate(vocab)}
C = np.zeros((len(vocab), len(vocab)))
for s in sentences:                       # 같은 문장에 함께 나온 횟수를 셉니다
    for a in s:
        for b in s:
            if a != b:
                C[ix[a], ix[b]] += 1

# 우연히 함께 나온 정도보다 얼마나 더 자주 나왔는지(PPMI)로 바꾼 뒤, 숫자 DIM개로 줄입니다 (SVD)
DIM = 50   # ← 좌표의 숫자 개수
total = C.sum()
ppmi = np.maximum(np.log((C * total) / (C.sum(1, keepdims=True) * C.sum(0, keepdims=True)) + 1e-12), 0)
U, S, Vt = np.linalg.svd(ppmi)
emb = U[:, :DIM] * S[:DIM]
emb /= np.linalg.norm(emb, axis=1, keepdims=True)
E = {w: emb[ix[w]] for w in WORDS}

print(f"낱말마다 숫자 {DIM}개짜리 좌표가 생겼습니다. 앞의 8개만 보면:")
for w in ["왕", "여왕", "사과"]:
    print(f"  '{w}':", np.round(E[w][:8], 2))

## 5단계. 임베딩 지도 그리기

숫자 50개짜리 좌표는 머릿속에 그릴 수 없습니다. 그래서 '가까운 낱말은 지도에서도 가까이' 놓이도록 평면에 펼쳐 그립니다(t-SNE라는 방법입니다).
뜻이 비슷한 말끼리 모여 있는지 보세요(그림 4-5).

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

names = list(WORDS)
xy = TSNE(n_components=2, perplexity=6, metric="cosine", init="pca",
          random_state=0).fit_transform(np.array([E[w] for w in names]))
colors = {"사람": "#e8743b", "도시·나라": "#5b8def", "과일": "#2ca02c", "동물": "#9467bd", "탈것": "#8c564b"}

plt.figure(figsize=(9, 7))
for (x, y), w in zip(xy, names):
    g = WORDS[w][0]
    plt.scatter(x, y, color=colors[g], s=60)
    plt.annotate(label(w), (x, y), textcoords="offset points", xytext=(4, 4), fontsize=10)
for g, c in colors.items():
    plt.scatter([], [], color=c, label=label(g))
plt.legend()
plt.title(label("임베딩 지도") if KFONT else "Embedding map")
plt.xticks([]); plt.yticks([])
plt.show()

### 가장 가까운 낱말 찾기

두 좌표가 같은 방향을 가리킬수록 뜻이 가깝습니다(코사인 유사도, 1에 가까울수록 비슷).

👉 **바꿔 볼 곳**: `QUERY`에 위 낱말 가운데 하나를 넣어 보세요.

In [ ]:
QUERY = "고양이"   # ← 다른 낱말을 넣어 보세요

def nearest(vec, exclude=(), k=4):
    vec = vec / np.linalg.norm(vec)
    sims = sorted(((float(E[w] @ vec), w) for w in WORDS if w not in exclude), reverse=True)
    return sims[:k]

for sim, w in nearest(E[QUERY], exclude={QUERY}):
    print(f"  {w:5s} 유사도 {sim:.2f}")

## 6단계. '왕 − 남자 + 여자'를 직접 계산하기

책 3-3절의 유명한 계산입니다. '왕'의 좌표에서 '남자'를 빼고 '여자'를 더한 자리에 가장 가까운 낱말은 무엇일까요?
계산에 쓴 세 낱말은 답에서 뺍니다.

👉 **바꿔 볼 곳**: `A`, `B`, `C`를 바꿔 '서울 − 한국 + 일본', '왕자 − 소년 + 소녀' 등도 해 보세요.

In [ ]:
A, B, C = "왕", "남자", "여자"   # ← A − B + C

result = E[A] - E[B] + E[C]
print(f"{A} − {B} + {C} ≈ ?")
for sim, w in nearest(result, exclude={A, B, C}, k=3):
    print(f"  {w:5s} 유사도 {sim:.2f}")

for a, b, c in [("서울", "한국", "일본"), ("왕자", "소년", "소녀"), ("파리", "프랑스", "중국")]:
    best = nearest(E[a] - E[b] + E[c], exclude={a, b, c}, k=1)[0][1]
    print(f"{a} − {b} + {c} ≈ {best}")

사람 낱말 네 개씩만 따로 지도에 그리고, '남자→여자'와 '왕→여왕' 화살표를 그려 봅니다. 두 화살표가 거의 나란하다면, 좌표 안에 '성별'이라는 방향이 생긴 것입니다(그림 4-6).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, four in zip(axes, [["남자", "여자", "왕", "여왕"], ["소년", "소녀", "왕자", "공주"]]):
    pxy = PCA(n_components=2, random_state=0).fit_transform(np.array([E[w] for w in four]))
    P = dict(zip(four, pxy))
    for w, (x, y) in P.items():
        ax.scatter(x, y, color="#e8743b", s=60)
        ax.annotate(label(w), (x, y), textcoords="offset points", xytext=(6, 6), fontsize=12)
    for a_, b_ in [(four[0], four[1]), (four[2], four[3])]:
        ax.annotate("", xy=P[b_], xytext=P[a_], arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))
    ax.set_xticks([]); ax.set_yticks([])
    ax.margins(0.25)
axes[0].set_title("man -> woman,  king -> queen")
axes[1].set_title("boy -> girl,  prince -> princess")
plt.show()

## 7단계. 주사위 언어 모델 만들기 (실습 2)

실습 2에서 종이와 주사위로 한 일을 컴퓨터로 합니다. 아래는 짧은 동화입니다.

1. 글에서 낱말마다 바로 다음에 나온 낱말을 모두 세어 표로 만듭니다.
2. 횟수에 비례하는 확률로 다음 낱말을 뽑습니다(주사위).
3. 고른 낱말 뒤에 다시 다음 낱말을 뽑습니다.

먼저 '토끼가' 다음에 무엇이 몇 번 나왔는지 봅니다. 책의 예처럼 깡충 3번, 잠을 2번, 웃었다 1번입니다.

In [ ]:
STORY = """
옛날 숲속에 토끼 한 마리가 살았다 .
토끼가 깡충 뛰었다 .
거북이가 천천히 걸었다 .
토끼가 깡충 뛰어 언덕에 올랐다 .
거북이가 토끼에게 경주를 하자고 말했다 .
토끼가 웃었다 .
경주가 시작되자 토끼가 깡충 달려 나갔다 .
거북이는 천천히 걸었다 .
언덕 위에서 토끼가 잠을 잤다 .
거북이는 쉬지 않고 걸었다 .
해가 지자 토끼가 잠을 깼다 .
거북이는 벌써 결승선에 도착했다 .
숲속 친구들이 거북이에게 박수를 쳤다 .
토끼는 다음부터 쉬지 않기로 했다 .
"""   # ← 다른 동화나 노래 가사로 바꿔 볼 수 있습니다 (낱말 사이를 띄어 쓰세요)

tokens = STORY.split()

def build_table(tokens, n_context):
    """앞 낱말 n_context개를 보고, 다음 낱말이 몇 번 나왔는지 셉니다."""
    table = {}
    for i in range(n_context, len(tokens)):
        ctx = tuple(tokens[i - n_context:i])
        table.setdefault(ctx, Counter())[tokens[i]] += 1
    return table

table1 = build_table(tokens, 1)
counts = table1[("토끼가",)]
print("'토끼가' 다음에 나온 낱말:", dict(counts))

face = 1                                    # 횟수를 주사위 눈으로 바꿉니다
for word, c in counts.most_common():
    faces = f"{face}" if c == 1 else f"{face}~{face + c - 1}"
    print(f"  주사위 {faces} → {word}")
    face += c

이제 글을 만들어 봅니다. 두 모델을 비교합니다.

- **앞 낱말 하나**만 보는 모델 (실습 2와 같음)
- **앞 낱말 셋**을 보는 모델 (표에서 앞 세 낱말이 같은 자리를 찾습니다. 그런 자리가 없으면 둘, 하나로 줄여서 찾습니다)

`TEMPERATURE`는 3-1절의 **온도**입니다. 1이면 주사위 그대로, 0이면 늘 가장 많이 나온 낱말만 고릅니다.
마지막 열은 만든 문장이 원래 글에 **그대로 있던 문장인지** 확인합니다.

👉 **바꿔 볼 곳**: `START`(첫 낱말), `TEMPERATURE`, `SEED`(주사위를 새로 굴리기)

In [ ]:
START = "토끼가"      # ← 첫 낱말 (동화에 있는 낱말)
LENGTH = 10           # ← 만들 낱말 수
TEMPERATURE = 1.0     # ← 0이면 '가장 많은 것만', 1이면 주사위 그대로, 2면 더 엉뚱하게
SEED = 0              # ← 숫자를 바꾸면 주사위를 새로 굴립니다

tables = {n: build_table(tokens, n) for n in (1, 2, 3)}

def next_word(history, max_context, rng, temperature):
    for n in range(max_context, 0, -1):          # 긴 문맥부터 찾고, 없으면 줄입니다
        ctx = tuple(history[-n:])
        if len(history) >= n and ctx in tables[n]:
            counter = tables[n][ctx]
            break
    else:
        return rng.choice(tokens)
    words = list(counter)
    if temperature == 0:
        return counter.most_common(1)[0][0]
    weights = [counter[w] ** (1 / temperature) for w in words]
    return rng.choices(words, weights=weights)[0]

def generate(max_context, seed, temperature):
    rng = random.Random(seed)
    out = [START]
    while len(out) < LENGTH:
        out.append(next_word(out, max_context, rng, temperature))
    return out

def in_story(words):
    text = " " + " ".join(tokens) + " "
    return (" " + " ".join(words) + " ") in text

for n in (1, 3):
    print(f"[앞 낱말 {n}개를 보는 모델, 온도 {TEMPERATURE}]")
    for k in range(3):
        words = generate(n, SEED + k, TEMPERATURE)
        print(f"  {' '.join(words)}    ← 원래 글에 그대로 있음: {'예' if in_story(words) else '아니오'}")
    print()

앞 낱말 하나만 보면 문장이 금방 엉뚱해집니다. 앞을 셋 보면 말은 훨씬 자연스러워지지만, 이렇게 작은 글에서는 원래 글을 거의 그대로 따라 합니다. 외운 글을 되풀이하는 셈입니다.
그래서 진짜 언어 모델은 엄청나게 많은 글로 배우고, 앞의 수천 토큰을 한꺼번에 살피는 어텐션(3-4절)을 씁니다.

## 생각해 볼 것

1. 같은 내용인데 한국어가 토큰을 더 많이 쓰면 한국어 사용자에게 어떤 손해가 생길까요? 4-3절의 '한국어로 생각하는 AI'와 연결해 생각해 보세요.
2. '왕 − 남자 + 여자 ≈ 여왕'은 컴퓨터가 성별의 뜻을 이해했다는 뜻일까요? 이 노트북의 임베딩이 무엇에서 만들어졌는지 떠올려 보세요.
3. 주사위 모델이 만든 '원래 글에 없던 그럴듯한 문장'은 창작일까요, 환각일까요? 3-7절의 환각과 비교해 보세요.